# DNM hypermutation pipeline: source merging, window mapping, and rate calculation

Merges de novo mutation (DNM) calls from multiple published sources into a
single, deduplicated, genome-build-consistent master file, then maps that
master set onto a genomic feature of interest (e.g. transcription start
sites) to compute position-resolved, trio-normalized mutation rates.

## Pipeline overview

| Part | What it does | Key output |
|---|---|---|
| **I** | Merge seven DNM sources (deCODE + six published studies) into one GRCh38-consistent, deduplicated file | `dnm_master_merged.txt` |
| **II** | Map every merged DNM onto a genomic feature's 1kb windows (position relative to the feature anchor) | `<region>_mapped.txt` |
| **III** | Classify each mapped mutation by substitution type, splitting C>T and G>A by CpG context | `<region>_mapped_count.csv` |
| **IV** | Compute per-position mutation rates (per-type diagnostic, and a trio-normalized headline rate), and plot | `<region>_overall_mutrate.tsv`, figures |

## Data sources merged in Part I

| Source | Trios | Reference |
|---|---|---|
| deCODE (Iceland) | 9,652 (direct count; see Part I, Section 9) | Palsson et al. 2025, *Nature* 639:700–707 |
| An et al. 2018 | 3,804 | An et al. 2018, *Science* 362:eaat6576 |
| Yuen et al. 2017 | ~2,624 | Yuen et al. 2017, *Nat. Neurosci.* 20:602–611 |
| Richter et al. 2020 (cases) | 749 | Richter et al. 2020, *Nat. Genet.* 52:769–777 |
| Goldmann et al. 2016 | 816 | Goldmann et al. 2016, *Nat. Genet.* 48:935–939 |
| Sasani et al. 2019 | see Part I, Section 9 caveat | Sasani et al. 2019, *eLife* 8:e46922 |
| Francioli et al. 2016 | 258 | Francioli et al. 2016, *Eur. J. Hum. Genet.* 23:1473–1480 |

Richter et al. 2020's unaffected-control cohort is drawn from the Simons
Simplex Collection — the same resource An et al. 2018 uses — and is excluded
from the merge (Part I, Section 7b) to avoid double-counting overlapping
samples.

## Requirements

- Python: `pandas`, `numpy`, `matplotlib`, `openpyxl`, `pyfaidx`, `pyliftover`
- `bedtools` (used indirectly if extending Part II to other extraction steps) — not required for the core pipeline as written
- `samtools` (Part III, reference-context lookup)
- A local GRCh38 reference FASTA (indexed) — path set in the configuration cell below

## Configuration

All file paths for the whole notebook — both the source-merging stage
(Part I) and the region-mapping/rate stage (Parts II–IV) — are defined once,
in the configuration cell immediately below. To re-run Parts II–IV against a
different genomic feature (a different promoter set, a different gene
class), change `region_name` and `region_bed` there; nothing else in Parts
II–IV needs editing.

**One path is repeated across cells rather than centralized**: every code
cell independently reloads its configuration from
`CONFIG["workdir"] + "/merge_config.json"` (an absolute path, hardcoded once
per cell). This is deliberate — it keeps every cell independently
re-runnable after a kernel restart without depending on another cell's
in-memory state — but it does mean that moving `WORKDIR` to a different
machine requires a find-and-replace across the notebook for that one path
string, not just an edit to the configuration cell.


# Part I: Merge external DNM sources into decode_dnm.txt's format

This is a **two-pass notebook, not a run-all-cells notebook**. Each of the six
external supplementary tables merged here is released in its own ad hoc format,
so this cannot be fully pre-configured the way the region-mapping stage is. The
workflow is:

1. **Pass 1** -- run Sections 1-2 (discovery). They print every file found in
   each folder, with a preview, and do nothing else.
2. **Read the output.** Figure out, for each dataset, which columns are chrom /
   pos / ref / alt / sample ID / parent-of-origin, what the delimiter and header
   row are, and what genome build the paper says it used (a starting guess only
   -- Section 4 checks this empirically rather than trusting it).
3. **Fill in Section 3's config** using what you found. It's stubbed out with
   placeholders and comments, not real values -- every `col_*` entry needs to
   be corrected to match what Section 1 actually printed for that file.
4. **Pass 2** -- run Sections 4 onward, which handle assembly detection,
   liftover, overlap detection, and the merge, generically, off whatever config
   you filled in.

## What each stage does and why

- **Section 5 (assembly detection)**: Samples positions from each dataset, looks up the reference base at
  that position in both an hg19 and an hg38 FASTA, and reports which build's
  reference the file's own `ref` column actually matches. This is the same
  check you'd do by hand -- automated and run on every row you sample, not a
  handful of spot checks.
- **Section 6 (liftover)**: `pyliftover`, not the UCSC binary -- pure Python, no
  external install. After lifting, it re-checks the ref allele against the
  hg38 FASTA and **drops** any site where it no longer matches, exactly the
  validation Guzman et al. describe doing for their own hg38->hg19 liftover
  step (a site must uniquely map *and* the reference identity must be
  unchanged, or it's dropped rather than kept with a silently wrong ref).
- **Section 7 (overlap)**: exact `(chrom, pos, ref, alt)` matches between every
  pair of datasets, once everything is on the same build. At de novo mutation
  rates (~1-2e-8/site/generation), any non-trivial count of exact matches
  between two independent studies is far beyond what coincidental recurrent
  mutation would produce -- it means the two studies share samples, not that
  two unrelated individuals independently mutated the identical base.
- **Section 8 (merge + dedup)**: drops exact-tuple duplicates (keeping the
  first occurrence, recording every source that reported it), and writes a
  master file in decode_dnm.txt's own column layout plus a `source` column.

## 0. Configuration

Only **one** reference FASTA is required -- GRCh38, the same Ensembl toplevel
FASTA used throughout the earlier pipeline. Assembly detection (Section 5)
checks each dataset against GRCh38 only: a low match rate means "assume
GRCh37", and liftover (Section 6) only needs the chain file below, not a
GRCh37 sequence, to do the actual coordinate conversion.

`chain_file`: leave `None` to let `pyliftover` auto-download and cache the
hg19->hg38 chain from UCSC on first use (works fine if this machine has normal
internet access). If it doesn't, download
`hg19ToHg38.over.chain.gz` from
`https://hgdownload.soe.ucsc.edu/goldenPath/hg19/liftOver/` yourself and set
`chain_file` to that path.

In [ ]:
import json, os

WORKDIR = "/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined"
REF_FASTA = "/media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/Homo_sapiens.GRCh38.dna.toplevel.fa"
os.makedirs(WORKDIR, exist_ok=True)

CONFIG = {
    "workdir": WORKDIR,
    "target_build": "hg38",   # matches decode_dnm.txt's own build throughout this pipeline
    "hg38_fasta": REF_FASTA,
    # No GRCh37 FASTA needed: the build check (Part I, Section 5) tests only
    # against GRCh38. A low match rate there means "assume GRCh37, liftover"
    # -- liftover itself only needs the chain file below, not a source-build
    # sequence, and the post-liftover ref re-check against GRCh38 is what
    # actually confirms the GRCh37 assumption was correct.
    "chain_file": None,  # None = auto-download and cache via pyliftover; else path to hg19ToHg38.over.chain.gz
    "build_match_threshold": 0.90,  # >= this on GRCh38 -> already GRCh38; below -> assume GRCh37, liftover

    # Part I source folders. decode_dnm.txt's folder is included here too,
    # purely so Part I Section 2 can print its column layout -- its GRCh38
    # build is already known and it skips assembly-detection/liftover
    # entirely; it's the reference schema Part I Section 8 aligns everything
    # else to.
    "folders": {
        "decode":       f"{WORKDIR}/../Decode",
        "Yuen":         f"{WORKDIR}/../Yuen_hg19",
        "An":           f"{WORKDIR}/../An",
        "Goldmann2016": f"{WORKDIR}/../Goldman_2016",
        "Richter":      f"{WORKDIR}/../Richter",
        "Francioli":    f"{WORKDIR}/../Francioli",
        "Sasani":       f"{WORKDIR}/../Sasani",
    },

    "n_sample_build_check": 2000,
    "output_master": os.path.join(WORKDIR, "dnm_master_merged.txt"),

    # ---- Parts II-IV: region mapping and rate calculation --------------
    # Swap these two to re-run Parts II-IV against a different genomic
    # feature set; everything downstream derives its filenames from
    # region_name automatically.
    "region_name": "lncRNA_noexpress",
    "region_bed": "/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/Rare_SNP/lncRNA/results/lncrna_non_testis_tss_hg38_final.txt",
    "nucleotide_counts_path": "/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/fantom5_lncRNA_noexpress_1kb_nucleotide_counts.txt",

    # Combined trio total from Part I, Section 9 -- the denominator for the
    # trio-normalized rate in Part IV. Update this if the merge or the
    # Section 7b exclusion list changes.
    "n_trios": 17339,
    "rolling_window": 100,
    "anchor": 500,   # Position - anchor = distance from the feature's anchor point (e.g. TSS), in bp
}

json.dump(CONFIG, open(os.path.join(WORKDIR, "merge_config.json"), "w"), indent=2)
print(f"Config -> {WORKDIR}/merge_config.json\n")
for key in ("hg38_fasta", "region_bed", "nucleotide_counts_path"):
    p = CONFIG[key]
    print(f"  {'OK ' if os.path.exists(p) else 'MISSING'}  {key}: {p}")
for name, path in CONFIG["folders"].items():
    print(f"  {'OK ' if os.path.isdir(path) else 'MISSING'}  folders[{name}]: {path}")


## 1. Shared library

In [ ]:
import json, os, sys, importlib

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))

LIB = r"""
import os, io, gzip, csv, math
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

BASES = "ACGT"
COMP = {"A": "T", "T": "A", "C": "G", "G": "C", "N": "N"}


# ------------------------------------------------------------- discovery
def _preview_text(path, n=6, max_cols_show=12):
    opener = gzip.open if path.endswith(".gz") else open
    try:
        with opener(path, "rt", errors="replace") as f:
            lines = [f.readline().rstrip("\n") for _ in range(n)]
    except Exception as e:
        return f"  !! could not read as text: {e}"
    out = []
    for i, line in enumerate(lines):
        if not line:
            continue
        # guess delimiter for display only
        for delim in ("\t", ",", " "):
            if delim in line:
                parts = line.split(delim)
                break
        else:
            parts = [line]
        shown = parts[:max_cols_show]
        more = f"  (+{len(parts) - max_cols_show} more cols)" if len(parts) > max_cols_show else ""
        out.append(f"  [{i}] ({len(parts)} fields) " + " | ".join(shown) + more)
    return "\n".join(out)


def _preview_xlsx(path, n=6):
    try:
        import openpyxl
        wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    except Exception as e:
        return f"  !! could not open as xlsx: {e}"
    out = [f"  sheets: {wb.sheetnames}"]
    for sheet in wb.sheetnames:
        ws = wb[sheet]
        try:
            dims = ws.calculate_dimension()
        except Exception:
            dims = "?"
        out.append(f"  --- sheet '{sheet}' ({dims}) ---")
        for i, row in enumerate(ws.iter_rows(min_row=1, max_row=n, values_only=True)):
            out.append(f"    [{i}] {row}")
    return "\n".join(out)


def discover_single_file(path):
    # Like discover_folder, but for a folder that contains files OTHER than
    # the one you actually want (e.g. decode_dnm.txt sitting alongside
    # unrelated files) -- previews exactly this one file, not everything
    # next to it.
    print(f"\n{'='*70}\n{path}\n{'='*70}")
    if not os.path.exists(path):
        print(f"  !! not found: {path}")
        return
    size = os.path.getsize(path)
    print(f"  ({size:,} bytes)")
    if path.lower().endswith((".xlsx", ".xls")):
        print(_preview_xlsx(path))
    else:
        print(_preview_text(path))


def discover_folder(folder, max_files=30):
    # Lists every file in `folder` (recursively, one level of subfolders) and
    # prints a short preview of each, dispatched by extension. This is meant
    # to be read by a human before writing the column-mapping config in the
    # next section -- it does not try to guess column meaning.
    print(f"\n{'='*70}\n{folder}\n{'='*70}")
    if not os.path.isdir(folder):
        print(f"  !! not a directory: {folder}")
        return
    found = []
    for root, dirs, files in os.walk(folder):
        for fn in files:
            found.append(os.path.join(root, fn))
    if not found:
        print("  (empty)")
        return
    for path in sorted(found)[:max_files]:
        size = os.path.getsize(path)
        print(f"\n- {os.path.relpath(path, folder)}  ({size:,} bytes)")
        ext = path.lower()
        if ext.endswith((".xlsx", ".xls")):
            print(_preview_xlsx(path))
        elif ext.endswith((".vcf", ".vcf.gz")):
            print(_preview_text(path, n=30))  # VCF headers are long; show more
        else:
            print(_preview_text(path))
    if len(found) > max_files:
        print(f"\n  ... and {len(found) - max_files} more files not shown")


# --------------------------------------------------------- chrom naming
def normalize_chrom(c, style):
    # style: 'plain' (Ensembl-like: '1','X','MT') or 'chr' (UCSC-like: 'chr1','chrX','chrM')
    c = str(c).strip()
    bare = c[3:] if c.lower().startswith("chr") else c
    if bare.upper() in ("MT", "M"):
        bare = "MT" if style == "plain" else "M"
    if style == "chr":
        return "chr" + bare
    return bare


def detect_chrom_style(keys):
    keyset = set(str(k) for k in keys)
    if "1" in keyset:
        return "plain"
    if "chr1" in keyset:
        return "chr"
    # fallback only if chr1 isn't decisively present either way
    return "chr" if any(str(k).lower().startswith("chr") for k in keyset) else "plain"


# ---------------------------------------------------------- fasta lookup
class FastaLookup:
    # Thin wrapper over pyfaidx with chrom-name-style auto-detection, so
    # callers can query with either 'chr1' or '1' regardless of how the
    # reference FASTA itself names its sequences.
    def __init__(self, path):
        from pyfaidx import Fasta
        self.fa = Fasta(path, sequence_always_upper=True, rebuild=True)
        self.style = detect_chrom_style(self.fa.keys())

    def base_at(self, chrom, pos_1based):
        # pos_1based: 1-based genomic coordinate, as used throughout DNM
        # mapping files in this project.
        key = normalize_chrom(chrom, self.style)
        if key not in self.fa:
            return None
        try:
            b = str(self.fa[key][pos_1based - 1:pos_1based]).upper()
            return b if b else None   # pyfaidx returns "" (not an exception) out of range
        except Exception:
            return None


# ------------------------------------------------------- dataset loading
def load_dataset(cfg, source_name):
    # Generic loader driven entirely by cfg -- no per-paper special-casing.
    # Required cfg keys: 'path', 'format' ('text'|'xlsx'), and either
    # 'col_chrom'/'col_pos'/'col_ref'/'col_alt' as column NAMES (if the file
    # has a header) or 0-based integer indices (if not). Optional:
    # 'delimiter' (default '\t'), 'header' (default 0, i.e. first row is
    # header; use None if headerless), 'sheet' (xlsx only), 'skiprows',
    # 'col_sample' (sample/individual ID column), 'col_inheritance'
    # (parent-of-origin column), 'assumed_build' (your best guess, verified
    # empirically in the next section -- not trusted blindly).
    fmt = cfg.get("format", "text")
    if fmt == "xlsx":
        df = pd.read_excel(cfg["path"], sheet_name=cfg.get("sheet", 0),
                           header=cfg.get("header", 0), skiprows=cfg.get("skiprows", None))
    else:
        df = pd.read_csv(cfg["path"], sep=cfg.get("delimiter", "\t"),
                         header=cfg.get("header", 0), skiprows=cfg.get("skiprows", None),
                         engine="python", dtype=str)

    def col(key):
        c = cfg.get(key)
        if c is None:
            return None
        if isinstance(c, int):
            return df.columns[c]
        return c

    out = pd.DataFrame()
    out["chrom"] = df[col("col_chrom")].astype(str)
    out["pos"] = pd.to_numeric(df[col("col_pos")], errors="coerce").astype("Int64")
    out["ref"] = df[col("col_ref")].astype(str).str.upper()
    out["alt"] = df[col("col_alt")].astype(str).str.upper()
    sc = col("col_sample")
    out["sample_id"] = df[sc].astype(str) if sc else "NA"
    ic = col("col_inheritance")
    out["inheritance"] = df[ic].astype(str) if ic else "unknown"
    out["source"] = source_name
    out["assumed_build"] = cfg.get("assumed_build", "unknown")

    n0 = len(out)
    out = out.dropna(subset=["chrom", "pos", "ref", "alt"])
    out = out[out["ref"].str.len() == 1]   # noindel: keep SNVs only, matching decode's own *_noindel convention
    out = out[out["alt"].str.len() == 1]
    out["pos"] = out["pos"].astype(int)
    dropped = n0 - len(out)
    print(f"[{source_name}] loaded {n0:,} rows -> {len(out):,} SNV rows kept "
          f"({dropped:,} dropped: missing fields or non-SNV)")
    return out.reset_index(drop=True)


# ------------------------------------------------ genome build detection
def check_ref_match(df, fasta, n_sample=2000, seed=0):
    # Single-build check, as requested: test against GRCh38 only. If the
    # match rate is high, the dataset IS GRCh38. If it's low, ASSUME GRCh37/
    # hg19 (no separate GRCh37 FASTA needed to confirm this -- liftover only
    # needs the chain file, not a source-build sequence) and let the
    # post-liftover ref re-check in liftover_df serve as the real
    # confirmation that hg19 was in fact the correct assumption.
    rng = np.random.default_rng(seed)
    n = len(df)
    idx = rng.choice(n, size=min(n_sample, n), replace=False) if n > 0 else np.array([], dtype=int)
    sample = df.iloc[idx]

    match, total, mismatches = 0, 0, []
    for _, row in sample.iterrows():
        ref_genome = fasta.base_at(row["chrom"], int(row["pos"]))
        if ref_genome is None:
            continue
        total += 1
        if ref_genome == row["ref"]:
            match += 1
        elif len(mismatches) < 5:
            mismatches.append((row["chrom"], row["pos"], row["ref"], ref_genome))
    rate = match / total if total else float("nan")
    return {"match": match, "total": total, "rate": rate, "examples": mismatches}


# -------------------------------------------------------------- liftover
def liftover_df(df, lo, target_fasta=None, verify_ref=True):
    # lo: a pyliftover.LiftOver instance already pointed at the right chain.
    # If target_fasta (a FastaLookup) is given and verify_ref=True, drops
    # any lifted site where the ref allele no longer matches the target
    # build -- same validation Guzman et al. describe for their own hg38->
    # hg19 liftover (site must uniquely map AND the reference base must be
    # unchanged, or the site is dropped).
    rows = []
    unmapped, multi, ref_changed = 0, 0, 0
    for _, row in df.iterrows():
        chrom_in = row["chrom"] if str(row["chrom"]).lower().startswith("chr") else "chr" + str(row["chrom"])
        hits = lo.convert_coordinate(chrom_in, int(row["pos"]) - 1)  # pyliftover is 0-based
        if not hits:
            unmapped += 1
            continue
        if len(hits) > 1:
            multi += 1
            continue
        new_chrom, new_pos0, strand, _ = hits[0]
        new_pos = new_pos0 + 1
        new_chrom_out = new_chrom[3:] if new_chrom.lower().startswith("chr") else new_chrom
        ref, alt = row["ref"], row["alt"]
        if strand == "-":
            ref, alt = COMP.get(ref, ref), COMP.get(alt, alt)
        if verify_ref and target_fasta is not None:
            true_ref = target_fasta.base_at(new_chrom_out, new_pos)
            if true_ref is not None and true_ref != ref:
                ref_changed += 1
                continue
        r = row.copy()
        r["chrom"], r["pos"], r["ref"], r["alt"] = new_chrom_out, new_pos, ref, alt
        rows.append(r)
    out = pd.DataFrame(rows).reset_index(drop=True) if rows else df.iloc[0:0].copy()
    report = {"in": len(df), "out": len(out), "unmapped": unmapped,
              "multi_mapped": multi, "ref_changed_after_lift": ref_changed}
    return out, report


def normalize_chrom_col(df, style="plain"):
    # Applied uniformly to every dataset before merge, lifted or not, so
    # datasets that pass through unchanged (already GRCh38, e.g. a "chr1"
    # naming file) end up in the same chrom-naming convention as datasets
    # that went through liftover_df (which always outputs plain style).
    # Without this, an un-lifted "chr1"-style dataset would silently mismatch
    # decode_dnm.txt's plain "1" convention in the final merged file.
    out = df.copy()
    out["chrom"] = out["chrom"].map(lambda c: normalize_chrom(c, style))
    return out


def load_decode(path, source_name="decode"):
    # decode_dnm.txt's actual, confirmed layout: Chr, pos, Ref>Alt, pid,
    # mut_orig, mut_class, with a header row -- different from every other
    # dataset here (mutation as one combined "Ref>Alt" field, not separate
    # ref/alt columns), so it gets its own loader rather than being forced
    # through load_dataset's column-mapping config.
    df = pd.read_csv(path, sep="\t", header=0, dtype=str)
    split = df["Ref>Alt"].str.split(">", n=1, expand=True)
    out = pd.DataFrame({
        "chrom": df["Chr"].astype(str),
        "pos": pd.to_numeric(df["pos"], errors="coerce").astype("Int64"),
        "ref": split[0].str.upper(),
        "alt": split[1].str.upper(),
        "sample_id": df.get("pid", "NA"),
        "inheritance": df.get("mut_orig", "unknown"),
        "source": source_name,
        "assumed_build": "hg38",
    })
    n0 = len(out)
    out = out.dropna(subset=["chrom", "pos", "ref", "alt"])
    out = out[(out["ref"].str.len() == 1) & (out["alt"].str.len() == 1)]
    out["pos"] = out["pos"].astype(int)
    print(f"[{source_name}] loaded {n0:,} rows -> {len(out):,} SNV rows kept "
          f"({n0 - len(out):,} dropped)")
    return out.reset_index(drop=True)


# ------------------------------------------------------- overlap checks
def pairwise_exact_overlap(datasets):
    # datasets: dict of {name: DataFrame} already on the SAME build, with
    # normalized chrom/pos/ref/alt. Reports exact (chrom,pos,ref,alt) match
    # counts between every pair. At de novo mutation rates (~1e-8/bp), any
    # non-trivial count here is far beyond chance and indicates shared
    # samples between the two studies, not coincidental recurrence -- unlike
    # a same-(chrom,pos)-different-allele match, which recurrent mutation at
    # a hypermutable site can produce legitimately and should NOT be treated
    # as a duplicate.
    keys = {}
    for name, df in datasets.items():
        keys[name] = set(zip(df["chrom"], df["pos"], df["ref"], df["alt"]))

    names = list(datasets.keys())
    rows = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            shared = keys[a] & keys[b]
            rows.append({"dataset_A": a, "dataset_B": b,
                        "n_A": len(keys[a]), "n_B": len(keys[b]),
                        "exact_overlap": len(shared),
                        "pct_of_A": 100 * len(shared) / max(len(keys[a]), 1),
                        "pct_of_B": 100 * len(shared) / max(len(keys[b]), 1)})
    return pd.DataFrame(rows).sort_values("exact_overlap", ascending=False).reset_index(drop=True)


def classify_mutation(ref, alt, chrom, pos, fasta):
    # Reverse-engineered directly from decode_dnm.txt's own mut_class
    # column, not assumed: CpG-ness is checked in genome-forward orientation
    # (ref==C with the next base G, or ref==G with the previous base C --
    # the two strand-equivalent readings of the same CpG dinucleotide), and
    # anything not CpG>TpG is folded to a pyrimidine-referenced pair (ref in
    # {A,G} -> complement both ref and alt). Verified against both real rows
    # from decode_dnm.txt (A>T -> T>A; C>T at a CpG -> CpG>TpG) before use.
    next_base = fasta.base_at(chrom, pos + 1)
    prev_base = fasta.base_at(chrom, pos - 1)
    is_cpg = (ref == "C" and next_base == "G") or (ref == "G" and prev_base == "C")
    if is_cpg and (ref, alt) in (("C", "T"), ("G", "A")):
        return "CpG>TpG"
    if ref in ("A", "G"):
        ref, alt = COMP[ref], COMP[alt]
    return f"{ref}>{alt}"


def normalize_inheritance(x):
    # decode_dnm.txt's own mut_orig values are lowercase 'father'/'mother'/
    # 'unknown'. External datasets vary (capitalized 'Father', literal 'NA',
    # NaN, etc.) -- normalized to match decode's convention rather than left
    # as a source-specific mess in the merged output.
    x = str(x).strip().lower()
    return x if x in ("father", "mother") else "unknown"


def dedup_master(df):
    # Drops exact (chrom,pos,ref,alt) duplicates, keeping the first
    # occurrence, and records every source that reported each kept variant
    # in a 'sources' column (so information about the overlap isn't
    # silently discarded, even though the duplicate rows are).
    key_cols = ["chrom", "pos", "ref", "alt"]
    sources_per_key = df.groupby(key_cols)["source"].apply(lambda s: ";".join(sorted(set(s))))
    df2 = df.drop_duplicates(subset=key_cols, keep="first").copy()
    df2 = df2.set_index(key_cols)
    df2["sources"] = sources_per_key
    df2 = df2.reset_index()
    n_dropped = len(df) - len(df2)
    return df2, n_dropped
"""

lib_path = os.path.join(CFG["workdir"], "merge_lib.py")
open(lib_path, "w").write(LIB)
sys.path.insert(0, CFG["workdir"])
import merge_lib; importlib.reload(merge_lib); m = merge_lib

print(f"Wrote {lib_path}")


## 2. Discovery -- PASS 1 STOPS HERE

Prints every file in every configured folder, with a preview -- except
`decode`, whose folder contains files other than the DNM list itself, so only
`decode_dnm.txt` is previewed there, not everything next to it. **Read this
output before writing anything in Section 3.** Nothing downstream can be
correct until the column mapping matches what's actually in these files.

In [ ]:
import json, os, sys, importlib

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
sys.path.insert(0, CFG["workdir"]); import merge_lib as m; importlib.reload(m)

for name, path in CFG["folders"].items():
    if name == "decode":
        # Decode's folder has files other than the DNM list itself --
        # preview only decode_dnm.txt, not everything sitting next to it.
        m.discover_single_file(os.path.join(path, "decode_dnm.txt"))
    else:
        m.discover_folder(path)


## 3. Per-dataset column mapping

Filled in from the actual files you uploaded for inspection, not placeholders
this time. **Still verify the paths match your real folder layout** -- these
assume each folder contains the file with the same name you uploaded (e.g.
`Yuen_hg19/Yuen_DNM.xlsx`); if your local filenames differ, fix the `path`
entries below before running Section 4.

Specific things confirmed directly from your files, not guessed:
- **Richter's two files are `.csv` by extension but tab-delimited**, and both
  have a title row before the real header (`header=1` skips it). Kept as two
  separate sources, `Richter_cases` and `Richter_controls`, rather than one
  merged "Richter" -- **`Richter_controls` carries a "Simons Family ID"
  column, direct confirmation these are Simons Simplex Collection
  individuals**, the same resource `An` draws from. This is exactly the
  overlap flagged in the intro, now with the column-level evidence in hand
  rather than just the paper's stated cohort description. Check
  `Richter_controls` vs `An` first in Section 7.
- **Goldmann's file has no per-individual sample ID column** (only
  `parentOfOrigin`) -- `col_sample` is `None` for this one, not an oversight.
- **Sasani and Yuen both use 0-based half-open `start`/`end` BED coordinates**,
  not a single position column. Verified directly (not assumed) that for
  every SNV row specifically, `end` equals the 1-based genomic position --
  confirmed on 4,298 + 23,386 Sasani SNVs and 5,000 Yuen SNVs, zero
  exceptions. `col_pos` is set to the `end`/`END` column accordingly. This
  does NOT hold for the indel rows mixed into these files, but indels are
  dropped by the SNV-only filter in `load_dataset` regardless, so it doesn't
  matter that `end` would be wrong for them.
- **An's and Yuen's sheets have a title row before the real header** (row 0 is
  a single merged-looking cell like "Table S2. List of de novo mutations."),
  hence `header=1` for both.
- **Sasani is two files for one source.** `paths` (plural) takes a list here;
  Section 4 loads and concatenates them under the single name `Sasani`,
  distinct from the Richter case where keeping two names was the point.

In [ ]:
import json, os, sys, importlib

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
F = CFG["folders"]

DATASET_CONFIG = {
    "Yuen": {
        "path": os.path.join(F["Yuen"], "Yuen_DNM.xlsx"),
        "format": "xlsx", "sheet": "Table S3", "header": 1,
        "col_chrom": "CHROM", "col_pos": "END", "col_ref": "REF", "col_alt": "ALT",
        "col_sample": "SAMPLE", "col_inheritance": None,
        "assumed_build": "hg19",   # folder name says hg19; verified empirically in Section 5 regardless
    },
    "An": {
        "path": os.path.join(F["An"], "An_DNM.xlsx"),
        "format": "xlsx", "sheet": "Table S2 de novo mutations", "header": 1,
        "col_chrom": "Chr", "col_pos": "Pos", "col_ref": "Ref", "col_alt": "Alt",
        "col_sample": "SampleID", "col_inheritance": None,
        "assumed_build": "unknown",
    },
    "Goldmann2016": {
        "path": os.path.join(F["Goldmann2016"], "Goldman_DNM.xlsx"),
        "format": "xlsx", "sheet": "observedMutations", "header": 0,
        "col_chrom": "Chromosome", "col_pos": "Start.position", "col_ref": "Reference", "col_alt": "Variant",
        "col_sample": None,   # no individual-ID column in this file
        "col_inheritance": "parentOfOrigin",
        "assumed_build": "hg19",
    },
    "Richter_cases": {
        "path": os.path.join(F["Richter"], "Richter_DNV_in_cases.csv"),
        "format": "text", "delimiter": "\t", "header": 1,   # row 0 is a title row, real header is row 1
        "col_chrom": "Chrom", "col_pos": "Pos", "col_ref": "Ref", "col_alt": "Alt",
        "col_sample": "Blinded ID", "col_inheritance": None,
        "assumed_build": "unknown",
    },
    "Richter_controls": {
        "path": os.path.join(F["Richter"], "Richter_SNV_in_control.csv"),
        "format": "text", "delimiter": "\t", "header": 1,
        "col_chrom": "Chrom", "col_pos": "Pos", "col_ref": "Ref", "col_alt": "Alt",
        "col_sample": "Blinded ID", "col_inheritance": None,
        "assumed_build": "unknown",
        # Simons Family ID column confirms these are SSC individuals -- same
        # resource as "An". Check this pair specifically in Section 7.
    },
    "Francioli": {
        "path": os.path.join(F["Francioli"], "Francioli_DNMs.txt"),
        "format": "text", "delimiter": "\t", "header": 0,
        "col_chrom": "CHROM", "col_pos": "POS", "col_ref": "REF", "col_alt": "ALT",
        "col_sample": "CHILD_ID", "col_inheritance": "ParentOfOrigin",
        "assumed_build": "hg19",
    },
    "Sasani": {
        # Two files, one source -- loaded and concatenated under this single
        # name in Section 4 (list, not a single path).
        "paths": [os.path.join(F["Sasani"], "Sasani_second_gen.dnms.txt"),
                 os.path.join(F["Sasani"], "Sasani_third_gen.dnms.txt")],
        "format": "text", "delimiter": "\t", "header": 0,
        "col_chrom": "chrom", "col_pos": "end", "col_ref": "ref", "col_alt": "alt",
        "col_sample": "new_sample_id", "col_inheritance": "phase",
        "assumed_build": "hg19",
    },
}

json.dump(DATASET_CONFIG, open(os.path.join(CFG["workdir"], "dataset_config.json"), "w"), indent=2)
print("Wrote dataset_config.json\n")
for name, c in DATASET_CONFIG.items():
    paths = c.get("paths", [c.get("path")])
    for p in paths:
        print(f"  [{name}] {p}")


## 4. Load each dataset

Loads every dataset through `DATASET_CONFIG`, and separately inspects
`decode_dnm.txt` itself (not through the merge machinery -- just to pin down
its exact columns, since Section 7 has to match that layout, not invent one).
SNV-only, matching decode's own `_noindel` convention -- indels are dropped
here, not silently mismapped.

In [ ]:
import json, os, sys, importlib

import pandas as pd

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
sys.path.insert(0, CFG["workdir"]); import merge_lib as m; importlib.reload(m)
DATASET_CONFIG = json.load(open(os.path.join(CFG["workdir"], "dataset_config.json")))
W = CFG["workdir"]

datasets = {}
for name, c in DATASET_CONFIG.items():
    paths = c["paths"] if "paths" in c else [c["path"]]
    missing = [p for p in paths if not os.path.exists(p)]
    if missing:
        for p in missing:
            print(f"[{name}] MISSING {p} -- fix Section 3 first")
        continue
    parts = [m.load_dataset({**c, "path": p}, name) for p in paths]
    datasets[name] = pd.concat(parts, ignore_index=True) if len(parts) > 1 else parts[0]
    if len(parts) > 1:
        print(f"[{name}] concatenated {len(paths)} files -> {len(datasets[name]):,} total SNV rows")

# decode_dnm.txt -- inspected, not loaded through the same pipeline. Look at
# this output and confirm it matches what Sections 6-7 assume about its
# layout (chrom/pos as the last two whitespace-separated fields, as in every
# other file this project has used so far). If it doesn't, Section 7's
# alignment step needs adjusting, not just this print.
decode_path = os.path.join(CFG["folders"]["decode"], "decode_dnm.txt")
if os.path.exists(decode_path):
    print(f"\n--- decode_dnm.txt first 3 rows ---")
    with open(decode_path) as f:
        for i, line in enumerate(f):
            if i >= 3: break
            print(" ", line.rstrip())
else:
    print(f"MISSING {decode_path}")


## 5. Genome assembly check -- GRCh38 first, assume GRCh37 otherwise

Single-FASTA check, as requested, rather than comparing against both builds.
For each dataset, samples up to `n_sample_build_check` rows and checks the ref
allele against GRCh38 at that position:

- **rate >= `build_match_threshold`** -> already GRCh38, no liftover needed.
- **rate < threshold** -> assumed GRCh37, sent to Section 6 for liftover. No
  separate GRCh37 FASTA is used to confirm this assumption up front -- the
  confirmation happens after liftover instead (Section 6's ref re-check
  against GRCh38): if a dataset was wrongly assumed GRCh37, liftover would
  shift its coordinates somewhere they don't belong, and the post-lift ref
  check would fail broadly rather than quietly succeeding.

A rate that lands well below `build_match_threshold` but still well above
chance (~0.3-0.5) usually means a chrom-naming mismatch, not really an
unknown build -- check `FastaLookup`'s auto-detected style against the file's
actual convention before concluding the build itself is the problem.

In [ ]:
import json, os, sys, importlib

import numpy as np

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
sys.path.insert(0, CFG["workdir"]); import merge_lib as m; importlib.reload(m)
W = CFG["workdir"]
THRESH = CFG["build_match_threshold"]

fasta_hg38 = m.FastaLookup(CFG["hg38_fasta"])

detected_build = {}
for name, df in datasets.items():
    r = m.check_ref_match(df, fasta_hg38, n_sample=CFG["n_sample_build_check"])
    build = "hg38" if r["rate"] >= THRESH else "hg19"
    detected_build[name] = build
    print(f"[{name}] (assumed: {DATASET_CONFIG[name]['assumed_build']})  "
          f"GRCh38 match: {r['match']:5d}/{r['total']:5d} ({r['rate']:.4f})  -> {build}")
    if 0.3 < r["rate"] < THRESH:
        print(f"    (below threshold, above chance -- assumed GRCh37, will liftover in Section 6)")
    elif r["rate"] <= 0.3 and r["total"] > 0:
        print(f"    !! match rate near chance level -- check chrom naming / column mapping, "
              f"this may not be a simple GRCh37-vs-GRCh38 issue")
        for ex in r["examples"]:
            print(f"       mismatch example: {ex}")

json.dump(detected_build, open(os.path.join(W, "detected_build.json"), "w"), indent=2)
print(f"\n-> detected_build.json")


## 6. Liftover to hg38

Only datasets `detected_build` marked as `hg19` get lifted; anything already
`hg38` passes through unchanged. Ref-allele re-validation is on by default
(`verify_ref=True`) -- sites where the lifted position's ref no longer matches
hg38 are dropped, not kept with a silently stale ref allele. This is also the
real confirmation of Section 5's GRCh37 assumption: a dataset that was
correctly assumed GRCh37 will show a high post-lift ref-match rate (few
`ref_changed_after_lift` drops); one that was mis-assumed will show heavy
loss here instead, which is the signal to go back and check its column
mapping rather than trust the liftover output.

Chrom naming is also normalized to plain style (`1`, not `chr1`) for
**every** dataset here, lifted or not -- `liftover_df` already outputs plain
style, but a pass-through GRCh38 dataset using `chr1`-style naming would
otherwise stay that way and silently mismatch decode_dnm.txt's own plain
convention in the final merge.

In [ ]:
import json, os, sys, importlib

from pyliftover.liftover import LiftOver

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
sys.path.insert(0, CFG["workdir"]); import merge_lib as m; importlib.reload(m)
W = CFG["workdir"]
detected_build = json.load(open(os.path.join(W, "detected_build.json")))

if CFG["chain_file"]:
    lo = LiftOver(CFG["chain_file"])
else:
    lo = LiftOver("hg19", "hg38")   # auto-downloads + caches on first use

fasta_hg38 = m.FastaLookup(CFG["hg38_fasta"])

datasets_hg38 = {}
for name, df in datasets.items():
    if detected_build.get(name) == "hg38":
        datasets_hg38[name] = m.normalize_chrom_col(df, style="plain")
        print(f"[{name}] already hg38, {len(df):,} rows unchanged (chrom naming normalized)")
        continue
    lifted, report = m.liftover_df(df, lo, target_fasta=fasta_hg38, verify_ref=True)
    datasets_hg38[name] = m.normalize_chrom_col(lifted, style="plain")
    print(f"[{name}] lifted: {report}")
    if report["in"] > 0 and report["out"] / report["in"] < 0.90:
        print(f"    !! lost more than 10% of rows in liftover -- check whether "
              f"'{name}' was really GRCh37, not just a chrom-naming or column issue")


## 7. Overlap detection

Exact `(chrom, pos, ref, alt)` matches between every pair, on hg38 throughout,
**including decode_dnm.txt itself** -- loaded fresh here via `load_decode`
(its own format, `Chr/pos/Ref>Alt/pid/mut_orig/mut_class`, doesn't fit the
`DATASET_CONFIG` column-mapping scheme the other six use). decode isn't added
to `datasets_hg38` or the merge itself -- it's the file this master is meant
to sit alongside, not something to fold into it -- this is purely to check
whether any external dataset happens to share samples with decode too, not
just with each other.

Check the Richter/An row specifically (see intro) before treating any other
row as noise.

In [ ]:
import json, os, sys, importlib

import pandas as pd

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
sys.path.insert(0, CFG["workdir"]); import merge_lib as m; importlib.reload(m)
W = CFG["workdir"]

decode_path = os.path.join(CFG["folders"]["decode"], "decode_dnm.txt")
decode_df = m.load_decode(decode_path)
decode_df = m.normalize_chrom_col(decode_df, style="plain")  # FIX: without this, decode's
# "chr1"-style chrom never matches datasets_hg38's normalized "1"-style chrom, so every
# overlap check against decode is guaranteed to read 0% regardless of true overlap

comparison_set = dict(datasets_hg38)
comparison_set["decode"] = decode_df

overlap_table = m.pairwise_exact_overlap(comparison_set)
overlap_table.to_csv(os.path.join(W, "pairwise_overlap.tsv"), sep="\t", index=False, float_format="%.3f")
pd.set_option("display.width", 160)
print(overlap_table.to_string(index=False))
print(f"\n-> pairwise_overlap.tsv")

flagged = overlap_table[overlap_table["exact_overlap"] > 0]
if len(flagged):
    print(f"\n{len(flagged)} pair(s) share exact DNM calls -- see above. Decide in "
          f"Section 7b which datasets to exclude outright (whole-dataset, to keep trio "
          f"counts well-defined) rather than letting Section 8's dedup silently mix "
          f"partial data from both sides of an overlapping pair.")
else:
    print("\nNo exact-tuple overlaps found between any pair, including decode.")


## 7b. Dataset exclusion decisions

**Whole-dataset exclusion, not row-level dedup**, for any pair with
substantial overlap. `dedup_master` (Section 8) only drops the exact
duplicate rows -- the non-overlapping remainder of BOTH datasets stays in
the merge, which means you'd have some trios' DNMs from dataset A and other
trios' DNMs from dataset B mixed into a set whose total trio count is no
longer a number you actually know. Excluding a dataset entirely here keeps
every kept dataset's own published trio count meaningful for a per-generation
rate calculation.

**An vs Richter_controls: 88,241 exact matches -- 80.9% of Richter_controls,
38.3% of An.** That's not a coincidental-recurrence signal, it's confirmation
that these are very substantially the same individuals (matching the
"Simons Family ID" column already found in Richter_controls in Section 3).
Keeping `An` (230,418 SNVs, larger) and dropping `Richter_controls` (109,038
SNVs) entirely is the default below -- change it if you'd rather keep
`Richter_controls` and drop `An` instead, but keep exactly one of the two,
not both.

Every other pair in Section 7's table is under 0.03% overlap in either
direction -- consistent with coincidental recurrent mutation at hypermutable
sites, not shared samples. Nothing else needs excluding on the current
evidence; revisit if Section 7's numbers change after you fix Sasani's path
and re-run.

In [ ]:
import json, os, sys, importlib

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
W = CFG["workdir"]

# Edit this list based on Section 7's overlap table before running Section 8.
EXCLUDED_DATASETS = [
    "Richter_controls",   # near-total overlap with An (SSC) -- An kept instead, see markdown
]

json.dump(EXCLUDED_DATASETS, open(os.path.join(W, "excluded_datasets.json"), "w"), indent=2)
print("Excluding from the merge entirely:")
for name in EXCLUDED_DATASETS:
    n = len(datasets_hg38[name]) if name in datasets_hg38 else "?"
    print(f"  [{name}]  {n:,} SNVs removed" if isinstance(n, int) else f"  [{name}]  (not currently loaded)")
if not EXCLUDED_DATASETS:
    print("  (none)")


## 8. Merge + dedup into decode_dnm.txt's actual format

decode_dnm.txt's real columns, confirmed directly (not assumed): `Chr, pos,
Ref>Alt, pid, mut_orig, mut_class`, `chr`-prefixed chrom naming, with a header
row. That's a different column order and a different chrom-naming convention
than the later window-mapped `decode_dnm_GRCh38_*_mapped_noindel.txt` files
elsewhere in this project use (those are plain `1`-style, no `chr` prefix,
relpos-first) -- two different conventions at two different pipeline stages,
both real, not a contradiction. This master file matches decode_dnm.txt
itself, since that's the file it's meant to sit alongside.

**`mut_class` is computed here, not carried over** -- none of the six external
datasets have an equivalent column, and decode_dnm.txt's own convention was
reverse-engineered from its two example rows, not guessed: a mutation is
`CpG>TpG` if the reference base is part of a CpG dinucleotide in either
strand orientation (ref=C with the next base G, or ref=G with the previous
base C) and the change is C>T or G>A; otherwise it's folded to a
pyrimidine-referenced pair (ref in {A,G} -> complement both ref and alt, e.g.
raw A>T becomes T>A). Verified against both of decode's real rows before use.
Runs one FASTA lookup per side per variant, so this is the slowest part of
the whole notebook on the full dataset -- expect it to take a while on
hundreds of thousands of rows.

`mut_orig` values are normalized to decode's own lowercase `father`/`mother`/
`unknown` convention -- external datasets vary (`Father`, `NA`, blank,
`NaN`), all mapped to `unknown` unless they're recognizably `father` or
`mother`.

Concatenates all hg38 datasets, drops exact-tuple duplicates (keeping first
occurrence, recording every source that reported each kept variant), computes
`mut_class`, normalizes `mut_orig`, and writes a `chr`-prefixed master file
with a header, column order matching decode_dnm.txt exactly plus `sources`
appended at the end.

In [ ]:
import json, os, sys, importlib

import pandas as pd

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
sys.path.insert(0, CFG["workdir"]); import merge_lib as m; importlib.reload(m)
W = CFG["workdir"]

EXCLUDED_DATASETS = json.load(open(os.path.join(W, "excluded_datasets.json")))
all_datasets = {"decode": decode_df, **datasets_hg38}
included = {k: v for k, v in all_datasets.items() if k not in EXCLUDED_DATASETS}
if EXCLUDED_DATASETS:
    print(f"Excluded per Section 7b: {EXCLUDED_DATASETS}")

master_raw = pd.concat(list(included.values()), ignore_index=True)
print(f"Pre-dedup total: {len(master_raw):,} SNVs across {len(included)} datasets "
      f"({len(all_datasets) - len(included)} excluded)")

master, n_dropped = m.dedup_master(master_raw)
print(f"Post-dedup total: {len(master):,}  ({n_dropped:,} exact-duplicate rows dropped)")

fasta_hg38 = m.FastaLookup(CFG["hg38_fasta"])
print("Computing mut_class (one FASTA lookup per neighbour per variant -- this is the slow step)...")
master["mut_class"] = [
    m.classify_mutation(r.ref, r.alt, r.chrom, int(r.pos), fasta_hg38)
    for r in master.itertuples()
]
master["mut_orig"] = master["inheritance"].map(m.normalize_inheritance)
master["mutation"] = master["ref"] + ">" + master["alt"]
master["chrom_out"] = master["chrom"].map(lambda c: m.normalize_chrom(c, "chr"))

# Matches decode_dnm.txt exactly: Chr, pos, Ref>Alt, pid, mut_orig, mut_class,
# plus sources appended.
out_cols = {"chrom_out": "Chr", "pos": "pos", "mutation": "Ref>Alt",
           "sample_id": "pid", "mut_orig": "mut_orig", "mut_class": "mut_class",
           "sources": "sources"}
master_out = master[list(out_cols.keys())].rename(columns=out_cols)

outpath = CFG["output_master"]
master_out.to_csv(outpath, sep="\t", index=False, header=True)
print(f"\n-> {outpath}")

print("\nmut_class distribution (post-dedup):")
print(master["mut_class"].value_counts().to_string())
print("\nrows by source dataset (post-dedup, counting multi-source rows once per source they belonged to):")
print(master["source"].value_counts().to_string())


## 9. Trio count validation

Cross-checks the number of unique sample IDs actually present in each
merged dataset against the trio count reported in that dataset's original
publication. Large deviations are worth investigating before trusting the
combined total as a per-generation-rate denominator; small deviations are
expected from SNV-only filtering, liftover loss, and cross-dataset
deduplication (Sections 6–8).

In [ ]:
import json, os
import pandas as pd

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
W = CFG["workdir"]
EXCLUDED_DATASETS = json.load(open(os.path.join(W, "excluded_datasets.json")))

# Published trio counts, for comparison against the direct unique-sample-ID
# count below. See the caveats above for decode and Sasani specifically.
LITERATURE_TRIOS = {
    "decode":           9652,  # Palsson et al. 2025 Nature -- direct-count-corroborated cohort size; see caveat above
    "An":               3804,  # An et al. 2018 Science: 1,902 quartets x 2 offspring (proband + unaffected sibling)
    "Yuen":             2624,  # Yuen et al. 2017 Nat Neurosci: 1,745 probands + 879 affected + 6 unaffected siblings
    "Richter_cases":     749,  # Richter et al. 2020 Nat Genet (abstract; supplementary methods state 763)
    "Richter_controls": 1611,  # Richter et al. 2020 Nat Genet -- excluded from merge (Section 7b), shown for reference only
    "Goldmann2016":      816,  # Goldmann et al. 2016 Nat Genet
    "Sasani":            434,  # different paper, same 33 CEPH-Utah families -- see caveat above
    "Francioli":         258,  # Francioli et al. 2016 Eur J Hum Genet
}

# Goldmann2016's file has no per-individual sample column (col_sample was
# None in Section 3) -- nunique() on it would just count the literal
# string "NA" once, which is not a real trio count.
NO_SAMPLE_ID = {"Goldmann2016"}

all_dfs = {"decode": decode_df, **datasets_hg38}

rows = []
for name, df in all_dfs.items():
    n_unique = None if name in NO_SAMPLE_ID else df["sample_id"].nunique()
    lit = LITERATURE_TRIOS.get(name)
    pct_diff = (100 * (n_unique - lit) / lit) if (n_unique is not None and lit) else None
    rows.append({"dataset": name, "n_dnms_in_file": len(df),
                 "unique_sample_ids": n_unique, "literature_trios": lit,
                 "pct_diff": pct_diff})

trio_compare = pd.DataFrame(rows).sort_values("dataset")
pd.set_option("display.width", 160)
print(trio_compare.to_string(index=False, float_format=lambda x: f"{x:+.1f}%" if isinstance(x, float) else str(x)))
trio_compare.to_csv(os.path.join(W, "trio_counts_compare.tsv"), sep="\t", index=False)

# Combined total for the datasets actually going into the merge: unique
# sample_id count where available, literature fallback for Goldmann2016.
included = trio_compare[~trio_compare["dataset"].isin(EXCLUDED_DATASETS)].copy()
included["trios_for_total"] = included["unique_sample_ids"].fillna(included["literature_trios"])
combined_total = int(included["trios_for_total"].sum())

print(f"\nExcluded from total: {EXCLUDED_DATASETS}")
print(f"Goldmann2016 uses its literature count ({LITERATURE_TRIOS['Goldmann2016']}) -- no sample_id column to count directly")
print(f"\nCombined trio total (denominator for Part IV's per-generation rate): {combined_total:,}")
print(f"-> trio_counts_compare.tsv")


# Part II: Map merged DNMs to genomic windows

Maps every mutation in `dnm_master_merged.txt` (Part I's output) onto the
1kb windows defined by `region_bed` (set in the configuration cell), and
records each mutation's position relative to that window's anchor (e.g.
distance from a TSS), together with its strand-based direction.

`region_bed` is expected in `chrom, start, end, direction` format, one row
per window. For minus-strand windows, `start` is the higher genomic
coordinate (the window's anchor side) and `end` the lower — not standard
ascending BED order. This is deliberate in the source files used
throughout this pipeline (confirmed directly against real window
coordinates), not a bug: it lets both strands use the same "anchor-relative
distance" formula shape.

Runs multi-process (one worker per chunk of `dnm_master_merged.txt`) since
the DNM set can be large; window lookup is a per-chromosome linear scan
across all windows on that chromosome for each mutation, which is the
dominant cost.


In [ ]:
import json, concurrent.futures, logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))


def read_region_file(file_path):
    """Read a region-window file (chrom, start, end, direction)."""
    data = {}
    try:
        with open(file_path, 'r') as file:
            for line in file:
                parts = line.strip().split()
                chr_num = parts[0].replace('chr', '')
                start, end = int(parts[1]), int(parts[2])
                direction = parts[3]
                data.setdefault(chr_num, []).append((start, end, direction))
    except Exception as e:
        logging.error(f"Error reading region file: {e}")
        raise
    return data


def process_chunk(chunk, region_data):
    """Map a chunk of dnm_master_merged.txt lines to region-relative positions."""
    results = []
    for line in chunk:
        try:
            parts = line.strip().split()
            chr_num = parts[0].replace('chr', '')
            coord = int(parts[1])
            # dnm_master_merged.txt columns after chrom/pos: Ref>Alt, pid,
            # mut_orig, mut_class, sources (5 fields) -- carried through
            # unchanged into the mapped output.
            extra = parts[2:]

            if chr_num in region_data:
                for start, end, direction in region_data[chr_num]:
                    if direction == '+' and start <= coord <= end:
                        results.append(f"{coord - start} {' '.join(extra)} {chr_num} {coord}\n")
                        break
                    elif direction == '-' and end <= coord <= start:
                        results.append(f"{start - coord} {' '.join(extra)} {chr_num} {coord}\n")
                        break
        except Exception as e:
            logging.error(f"Error processing line: {line.strip()}. Error: {e}")
    return results


def process_files(file1_path, region_path, output_file_path, chunk_size=100_000):
    region_data = read_region_file(region_path)
    try:
        with open(file1_path, 'r') as file1, open(output_file_path, 'w') as output_file:
            next(file1)   # skip dnm_master_merged.txt's header row

            with concurrent.futures.ProcessPoolExecutor() as executor:
                chunk, futures = [], []
                for line in file1:
                    chunk.append(line)
                    if len(chunk) == chunk_size:
                        futures.append(executor.submit(process_chunk, chunk, region_data))
                        chunk = []
                if chunk:
                    futures.append(executor.submit(process_chunk, chunk, region_data))
                for future in concurrent.futures.as_completed(futures):
                    output_file.writelines(future.result())

        logging.info("File processing completed successfully.")
    except Exception as e:
        logging.error(f"Error processing files: {e}")


region_name = CFG["region_name"]
mapped_path = f"{CFG['workdir']}/{region_name}_mapped.txt"

process_files(CFG["output_master"], CFG["region_bed"], mapped_path)
print(f"-> {mapped_path}")


# Part III: Classify mutations by substitution type and CpG context

For every mapped mutation, tallies the 12 canonical substitution types per
position (1–1000), and additionally splits `C>T` and `G>A` into CpG and
non-CpG context by looking up the neighbouring reference base (the base
after a mutated C, or before a mutated G) via `samtools faidx`. Reference
lookups are batched per chromosome to avoid one `samtools` call per
mutation.


In [ ]:
import json, subprocess, collections
import pandas as pd

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
region_name = CFG["region_name"]
mapped_path = f"{CFG['workdir']}/{region_name}_mapped.txt"
count_path = f"{CFG['workdir']}/{region_name}_mapped_count.csv"

with open(mapped_path) as f:
    lines = f.readlines()

mutation_types = ["A>G", "T>C", "C>G", "T>G", "C>A", "A>T",
                  "G>C", "G>T", "C>T", "T>A", "A>C", "G>A"]
extra_types    = ["C>T_CpG", "C>T_nonCpG", "G>A_CpG", "G>A_nonCpG"]
counts = {i: {m: 0 for m in mutation_types + extra_types} for i in range(1, 1001)}

# ── Pass 1: parse all lines, collect positions needing context lookup ──────
parsed   = []   # (row_index, mut, chrom, coord) for all valid lines
to_fetch = collections.defaultdict(set)   # chrom -> {positions to query}

for line in lines:
    parts = line.split()
    if not parts or parts[0] == 'NA':
        continue
    try:
        row_index = int(parts[0])
    except ValueError:
        continue
    if not (1 <= row_index <= 999) or len(parts) < 5:
        continue

    mut = parts[1]
    if mut not in mutation_types:
        continue

    chrom = parts[6]
    coord = int(parts[7])
    parsed.append((row_index, mut, chrom, coord))

    if mut == 'C>T':
        to_fetch[chrom].add(coord + 1)   # base after C, to check for G
    elif mut == 'G>A':
        to_fetch[chrom].add(coord - 1)   # base before G, to check for C

print(f"Parsed {len(parsed):,} mutations")
print(f"Positions to fetch: {sum(len(v) for v in to_fetch.values()):,} across {len(to_fetch)} chromosomes")

# ── Batched samtools: one call per chromosome ───────────────────────────────
base_lookup = {}   # (chrom, pos) -> reference base

for chrom, positions in sorted(to_fetch.items()):
    regions_file = f'/tmp/regions_{chrom}.txt'
    with open(regions_file, 'w') as rf:
        for pos in sorted(positions):
            rf.write(f"{chrom}:{pos}-{pos}\n")

    result = subprocess.run(
        ['samtools', 'faidx', CFG["hg38_fasta"], '-r', regions_file],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    if result.returncode != 0:
        print(f"samtools error on chrom {chrom}:\n{result.stderr[:300]}")
        continue

    out_lines = result.stdout.strip().split('\n')
    fetched = 0
    for i in range(0, len(out_lines) - 1, 2):
        header = out_lines[i]          # e.g. >1:959757-959757
        seq = out_lines[i + 1].upper() if i + 1 < len(out_lines) else 'N'
        try:
            pos = int(header.split(':')[1].split('-')[0])
            base_lookup[(chrom, pos)] = seq[0] if seq else 'N'
            fetched += 1
        except (IndexError, ValueError):
            continue

    print(f"  chr{chrom}: fetched {fetched:,} / {len(positions):,} positions")

print(f"\nTotal bases in lookup: {len(base_lookup):,}")

# ── Pass 2: count mutations, split C>T and G>A by CpG context ──────────────
for row_index, mut, chrom, coord in parsed:
    counts[row_index][mut] += 1
    if mut == 'C>T':
        next_base = base_lookup.get((chrom, coord + 1), 'N')
        counts[row_index]['C>T_CpG' if next_base == 'G' else 'C>T_nonCpG'] += 1
    elif mut == 'G>A':
        prev_base = base_lookup.get((chrom, coord - 1), 'N')
        counts[row_index]['G>A_CpG' if prev_base == 'C' else 'G>A_nonCpG'] += 1

# ── Save ─────────────────────────────────────────────────────────────────
df = pd.DataFrame.from_dict(counts, orient='index', columns=mutation_types + extra_types)
df.to_csv(count_path, index_label='Position')
print(f"\nCounts saved -> {count_path}")
print(df[['C>T', 'C>T_CpG', 'C>T_nonCpG', 'G>A', 'G>A_CpG', 'G>A_nonCpG']].sum())


# Part IV: Mutation rate calculation and plotting

Two distinct rate calculations, kept separate deliberately rather than
merged into one:

- **IV-A — per-mutation-type diagnostic rates.** Each of the 12
  substitution types (C>T and G>A split by CpG context), divided by the
  matching nucleotide count at that position (`fantom5_..._nucleotide_counts.txt`).
  **Not normalized by trio count** — these show the *relative shape* of
  each mutation type's positional profile, not an absolute per-generation
  rate, and are not comparable in magnitude to Part IV-B or to each other
  across mutation types with different overall base rates.
- **IV-B — trio-normalized headline rate.** All mutation types pooled
  (nonCpG-only and CpG-inclusive versions), divided by nucleotide count
  *and* by the combined trio total from Part I, Section 9, with a
  rolling-window-100 pooled-sum smoothing. This is the number to report as
  a per-generation, per-site mutation rate.


## IV-A. Per-mutation-type diagnostic rates (not trio-normalized)


In [ ]:
import json
import pandas as pd
import numpy as np

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
region_name = CFG["region_name"]
count_path = f"{CFG['workdir']}/{region_name}_mapped_count.csv"
rates_path = f"{CFG['workdir']}/{region_name}_mutation_rates.csv"

dnm_counts = pd.read_csv(count_path, index_col='Position')
nuc_counts = pd.read_csv(CFG["nucleotide_counts_path"], sep='\t', index_col='Position')

# Align on Position index (both should be 1-1000)
dnm_counts, nuc_counts = dnm_counts.align(nuc_counts, join='inner', axis=0)
print(f"Aligned positions: {len(dnm_counts)}")

# Denominator per mutation type. A/T mutations use total A or T (no CpG
# context possible); C and G mutations to C>T/G>A use the CpG-split
# denominator, matching the numerator's split exactly.
DENOM = {
    'A>G': 'A', 'A>T': 'A', 'A>C': 'A',
    'T>C': 'T', 'T>G': 'T', 'T>A': 'T',
    'C>T_nonCpG': 'C_not_in_CpG',
    'C>G': 'C',
    'C>A': 'C',
    'G>A_nonCpG': 'G_not_in_CpG',
    'G>T': 'G',
    'G>C': 'G',
}

rates = pd.DataFrame(index=dnm_counts.index)
for mut, denom_col in DENOM.items():
    if mut not in dnm_counts.columns:
        continue
    denom_vals = nuc_counts[denom_col].replace(0, np.nan)
    rates[mut] = dnm_counts[mut] / denom_vals

CpG_MUTATIONS = {
    'C>T_CpG': ('C>T_CpG', 'C_in_CpG'),
    'G>A_CpG': ('G>A_CpG', 'G_in_CpG'),
}
for rate_col, (mut_col, denom_col) in CpG_MUTATIONS.items():
    if mut_col not in dnm_counts.columns:
        continue
    denom_vals = nuc_counts[denom_col].replace(0, np.nan)
    rates[rate_col] = dnm_counts[mut_col] / denom_vals

rates.to_csv(rates_path, index_label='Position')
print(f"Mutation rates saved -> {rates_path}")
print("\nRate summary (mean across positions):")
print(rates.mean().to_string())


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
region_name = CFG["region_name"]
rates_path = f"{CFG['workdir']}/{region_name}_mutation_rates.csv"
count_path = f"{CFG['workdir']}/{region_name}_mapped_count.csv"
plots_dir = f"{CFG['workdir']}/{region_name}_plots"

rates = pd.read_csv(rates_path, index_col='Position')
positions = rates.index
os.makedirs(plots_dir, exist_ok=True)

# Each entry: (non-CpG col, CpG col or None, base color non-CpG, base color CpG)
MUTATIONS = {
    'A>G': ('A>G',        None,        '#4dabf7', None),
    'A>T': ('A>T',        None,        '#74c0fc', None),
    'A>C': ('A>C',        None,        '#a5d8ff', None),
    'T>C': ('T>C',        None,        '#f783ac', None),
    'T>G': ('T>G',        None,        '#faa2c1', None),
    'T>A': ('T>A',        None,        '#fcc2d7', None),
    'C>T': ('C>T_CpG',   'C>T_nonCpG', '#69db7c', '#2f9e44'),
    'C>G': ('C>G',        None,        '#8ce99a', None),
    'C>A': ('C>A',        None,        '#b2f2bb', None),
    'G>A': ('G>A_CpG',   'G>A_nonCpG', '#ffa94d', '#e67700'),
    'G>T': ('G>T',        None,        '#ffc078', None),
    'G>C': ('G>C',        None,        '#ffd8a8', None),
}

BG, TEXT, MUTED = '#0d1117', '#e9ecef', '#adb5bd'


def style_ax(ax):
    ax.set_facecolor(BG)
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
    ax.yaxis.get_offset_text().set_color(MUTED)
    ax.axvline(x=1, color='white', linewidth=0.8, linestyle='--', alpha=0.4)


def make_plot(pos, data_dict, title, filename, binned=False):
    """data_dict: {label: (values, color)}"""
    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_facecolor(BG)
    style_ax(ax)
    for label, (vals, color) in data_dict.items():
        ax.plot(pos, vals, linewidth=1.4 if binned else 1.0, alpha=0.9, color=color, label=label)
    ax.set_title(title, color=TEXT, fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel('Position relative to feature anchor (bp)', color=MUTED, fontsize=9)
    ax.set_ylabel('Mutation rate', color=MUTED, fontsize=9)
    ax.legend(fontsize=8, framealpha=0.2, labelcolor='white', facecolor='#1a1a2e', loc='upper right')
    plt.tight_layout()
    plt.savefig(filename, dpi=180, bbox_inches='tight', facecolor=BG)
    plt.show()
    plt.close()


def rolling_window(pos, vals, window=100):
    """Centered rolling mean of raw per-position ratios -- a display
    smoother only, distinct from Part IV-B's pooled-sum rolling rate."""
    s = pd.Series(vals, index=pos)
    rolled = s.rolling(window=window, center=True).mean()
    return rolled.index.values, rolled.values


# ── 12 individual mutation plots ────────────────────────────────────────
for mut_name, (col, cpg_col, color, cpg_color) in MUTATIONS.items():
    safe = mut_name.replace('>', '_')
    data = {}
    if col in rates.columns:
        data[f'{mut_name} (non-CpG)' if cpg_col else mut_name] = (rates[col].values, color)
    if cpg_col and cpg_col in rates.columns:
        data[f'{mut_name} (CpG)'] = (rates[cpg_col].values, cpg_color)

    make_plot(positions, data, title=f'{mut_name} mutation rate across {region_name} window',
              filename=f'{plots_dir}/{safe}_rate.png')

    data_rolled, rp = {}, positions
    for label, (vals, color_) in data.items():
        rp, rv = rolling_window(positions, vals, CFG["rolling_window"])
        data_rolled[label] = (rv, color_)
    make_plot(rp, data_rolled, title=f'{mut_name} mutation rate -- {CFG["rolling_window"]}bp rolling window',
              filename=f'{plots_dir}/{safe}_rate_rolled.png', binned=True)
    print(f'Saved {mut_name} plots')

# ── Overall: total mutation COUNT per position ──────────────────────────
BASE_MUTATIONS = ["A>G", "T>C", "C>G", "T>G", "C>A", "A>T", "G>C", "G>T", "C>T", "T>A", "A>C", "G>A"]
dnm_counts = pd.read_csv(count_path, index_col='Position')
total_counts = dnm_counts[BASE_MUTATIONS].sum(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG); style_ax(ax)
ax.plot(total_counts.index, total_counts.values, linewidth=0.8, color=TEXT, alpha=0.9)
ax.set_title(f'Total DNM count per position across {region_name} window', color=TEXT, fontsize=12, fontweight='bold')
ax.set_xlabel('Position relative to feature anchor (bp)', color=MUTED, fontsize=9)
ax.set_ylabel('DNM count', color=MUTED, fontsize=9)
plt.tight_layout()
plt.savefig(f'{plots_dir}/total_mutations.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); plt.close()

rp, rv = rolling_window(total_counts.index, total_counts.values, CFG["rolling_window"])
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG); style_ax(ax)
ax.plot(rp, rv, linewidth=1.4, color=TEXT, alpha=0.9)
ax.set_title(f'Total DNM count -- {CFG["rolling_window"]}bp rolling window', color=TEXT, fontsize=12, fontweight='bold')
ax.set_xlabel('Position relative to feature anchor (bp)', color=MUTED, fontsize=9)
ax.set_ylabel('Mean DNM count', color=MUTED, fontsize=9)
plt.tight_layout()
plt.savefig(f'{plots_dir}/total_mutations_rolled.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); plt.close()

# ── All 12 rates overlaid -- rolling window ─────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG); style_ax(ax)
for mut_name, (col, _, color, _) in MUTATIONS.items():
    if col in rates.columns:
        rp, rv = rolling_window(positions, rates[col].values, CFG["rolling_window"])
        ax.plot(rp, rv, linewidth=1.0, alpha=0.85, color=color, label=mut_name)
ax.set_title('All mutation rates -- rolling window', color=TEXT, fontsize=12, fontweight='bold')
ax.set_xlabel('Position relative to feature anchor (bp)', color=MUTED, fontsize=9)
ax.set_ylabel('Mutation rate', color=MUTED, fontsize=9)
ax.legend(fontsize=7, framealpha=0.2, labelcolor='white', facecolor='#1a1a2e', ncol=4, loc='upper right')
plt.tight_layout()
plt.savefig(f'{plots_dir}/all_mutations_rate_rolled.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); plt.close()

print(f"\nAll plots saved to {plots_dir}/")


## IV-B. Trio-normalized headline rate (nonCpG vs. all mutations)

Numerator and denominator are pooled separately over each rolling window
before taking one ratio per position — not a rolling mean of per-position
ratios, which would be a materially different (and less correct) statistic
at these count sizes. The nonCpG denominator excludes CpG-context C's and
G's, matching the numerator's exclusion exactly.


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CFG = json.load(open("/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/DNM/master_combined/merge_config.json"))
region_name = CFG["region_name"]
N_TRIOS = CFG["n_trios"]
WINDOW = CFG["rolling_window"]
ANCHOR = CFG["anchor"]

nuc = pd.read_csv(CFG["nucleotide_counts_path"], sep="\t")
dnm = pd.read_csv(f"{CFG['workdir']}/{region_name}_mapped_count.csv")

df = nuc.merge(dnm, on="Position", how="inner").sort_values("Position").reset_index(drop=True)
assert len(df) == 1000, f"expected 1000 positions, got {len(df)} -- check both files cover Position 1-1000"

ALL_TYPES = ["A>G", "T>C", "C>G", "T>G", "C>A", "A>T", "G>C", "G>T", "C>T", "T>A", "A>C", "G>A"]
NON_CT_GA = [t for t in ALL_TYPES if t not in ("C>T", "G>A")]

num_all = df[ALL_TYPES].sum(axis=1)
den_all = df["A"] + df["T"] + df["C"] + df["G"]

num_noncpg = df[NON_CT_GA].sum(axis=1) + df["C>T_nonCpG"] + df["G>A_nonCpG"]
den_noncpg = df["A"] + df["T"] + df["C_not_in_CpG"] + df["G_not_in_CpG"]


def rolling_bounds(window):
    if window % 2 == 0:
        return window // 2 - 1, window // 2
    h = (window - 1) // 2
    return h, h


def rolling_rate(num, den, window, n_trios):
    left, right = rolling_bounds(window)
    n = len(num)
    num_roll, den_roll = np.full(n, np.nan), np.full(n, np.nan)
    for i in range(n):
        lo, hi = i - left, i + right + 1
        if lo >= 0 and hi <= n:
            num_roll[i] = num.iloc[lo:hi].sum()
            den_roll[i] = den.iloc[lo:hi].sum()
    with np.errstate(divide="ignore", invalid="ignore"):
        rate = np.where(den_roll > 0, num_roll / den_roll / n_trios, np.nan)
    return rate


rate_all = rolling_rate(num_all, den_all, WINDOW, N_TRIOS)
rate_noncpg = rolling_rate(num_noncpg, den_noncpg, WINDOW, N_TRIOS)

out = pd.DataFrame({"Position": df["Position"], "RelPos": df["Position"] - ANCHOR,
                    "rate_all_incl_CpG": rate_all, "rate_nonCpG": rate_noncpg})
out_path = f"{CFG['workdir']}/{region_name}_overall_mutrate.tsv"
out.to_csv(out_path, sep="\t", index=False, float_format="%.6g")

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(out["RelPos"], out["rate_all_incl_CpG"], lw=1.6, color="#CC5471", label="All mutations (incl. CpG)")
ax.plot(out["RelPos"], out["rate_nonCpG"], lw=1.6, color="#0C8BBC", label="nonCpG only")
ax.axvline(0, color="0.6", ls=":", lw=1, zorder=0)
ax.set_title(f"{region_name} mutation rate per generation (rolling window={WINDOW}bp, n_trios={N_TRIOS:,})")
ax.set_xlabel("Position relative to feature anchor (bp)")
ax.set_ylabel("Mutations / site / generation")
ax.grid(alpha=0.25); ax.legend(frameon=False)
fig.tight_layout()
fig_path = f"{CFG['workdir']}/{region_name}_overall_mutrate.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"-> {out_path}\n-> {fig_path}")
